# Día 1 · Carga de datos Lending Club

**Objetivo:** dejar en `data/interim/` un archivo Parquet solo con los préstamos a **36 meses** originados entre **2012 y 2015**, con las columnas que usaremos.

**Por qué así:**
- El archivo original pesa ~1.6 GB y tiene 151 columnas. Leerlo completo puede agotar la memoria, así que lo leemos **por bloques** (`chunksize`) y con **`usecols`**.
- Solo 36 meses: los datos llegan hasta dic-2018, así que un préstamo a 36 meses de 2015 ya terminó y tiene su resultado observado. Uno a 60 meses no.
- Parquet: guarda los tipos de datos y carga 10–20 veces más rápido que CSV en los días siguientes.

Donde veas `# TODO` te toca completar a ti. Si te atascas más de 15 minutos, escríbeme.

In [1]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw/accepted_2007_to_2018Q4.csv.gz")   # ajusta si tu archivo quedó en otra ruta
OUT = Path("../data/interim/loans_36m_2012_2015.parquet")
OUT.parent.mkdir(parents=True, exist_ok=True)
print(RAW.exists(), RAW)

True ..\data\raw\accepted_2007_to_2018Q4.csv.gz


## 1. Mirar las columnas sin cargar los datos

`nrows=0` lee solo el encabezado. Guardamos la lista completa porque el **Día 3** la clasificaremos en permitidas y prohibidas (fuga de información).

In [2]:
cols_todas = pd.read_csv(RAW, nrows=0).columns.tolist()
print(len(cols_todas), "columnas")
pd.Series(cols_todas, name="columna").to_csv("../data/interim/columnas_originales.csv", index=False)
cols_todas[:20]

151 columnas


['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc']

## 2. Columnas que vamos a cargar

Todas son conocidas **al momento de aprobar** el crédito, más `loan_status` (el resultado) e `issue_d` (fecha de originación).

In [3]:
COLS = [
    "id", "issue_d", "term", "loan_status",
    # condiciones del préstamo
    "loan_amnt", "int_rate", "installment", "grade", "sub_grade", "purpose",
    # perfil del solicitante
    "emp_length", "home_ownership", "annual_inc", "verification_status", "addr_state",
    # buró de crédito
    "dti", "delinq_2yrs", "earliest_cr_line", "fico_range_low", "fico_range_high",
    "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc",
    "mort_acc", "pub_rec_bankruptcies",
]
faltan = set(COLS) - set(cols_todas)
print("Columnas que no existen:", faltan or "ninguna")

Columnas que no existen: ninguna


## 3. Leer por bloques y filtrar

Pistas:
- `term` viene como texto con espacio inicial: `" 36 months"`. Usa `.str.strip()`.
- `issue_d` viene como `"Dec-2015"`. Conviértelo con `pd.to_datetime(..., format="%b-%Y")`.
- Al final de cada bloque guarda solo las filas que cumplen ambos filtros.

In [ ]:
bloques = []
for i, chunk in enumerate(pd.read_csv(RAW, usecols=COLS, chunksize=200_000, low_memory=False)):
    # TODO 1: convierte chunk["issue_d"] a fecha
    chunk["issue_d"] = ...

    # TODO 2: crea una máscara booleana: plazo de 36 meses Y año de issue_d entre 2012 y 2015
    mask = ...

    bloques.append(chunk[mask])
    print(f"bloque {i}: {mask.sum():,} filas útiles")

df = pd.concat(bloques, ignore_index=True)
df.shape

## 4. Controles de calidad

Antes de guardar, verifica. Anota los resultados en `bitacora.md`.

In [ ]:
# TODO 3: ¿cuántos préstamos hay por año de originación? (pista: df["issue_d"].dt.year.value_counts())
...

In [ ]:
# TODO 4: ¿qué valores toma loan_status y cuántos hay de cada uno?
...

In [ ]:
# ¿Hay ids repetidos? Debe dar 0
print("ids duplicados:", df["id"].duplicated().sum())
# Memoria usada
print(f"memoria: {df.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

## 5. Guardar

In [ ]:
df.to_parquet(OUT, index=False)
print("guardado:", OUT, f"{OUT.stat().st_size / 1e6:,.1f} MB")
pd.read_parquet(OUT).head()

## Para la bitácora

Responde en `bitacora.md`:
1. ¿Cuántas filas quedaron y cuántas por año?
2. ¿Qué estados de `loan_status` aparecen? ¿Cuáles serán "malo", cuáles "bueno" y cuáles excluiremos mañana?
3. ¿Qué fue lo más difícil hoy?